# Decision Tree Classifier

This notebook trains and tunes a Decision Tree classifier for **student dropout and academic outcome prediction**.

It uses the shared training and test CSV files generated from the Power BI exported ML-ready flat table.

The saved split is reused without creating a new train/test split.

Categorical encoding, numeric scaling, and classification are combined in a Pipeline. During cross-validation, preprocessing is fitted only on the training portion of each fold.

The selected Pipeline is refitted on the full training split and
evaluated on the held-out test split.

## 1. Import libraries and locate project folders

In [1]:
from pathlib import Path
import json
import pandas as pd
import joblib

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Locate project root robustly whether the notebook is run from the project root or notebook folder.
cwd = Path.cwd()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data" / "4_model_inputs").exists() and (candidate / "_03_Models").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root. Run this notebook from inside the project folder.")

MODEL_INPUT_DIR = PROJECT_ROOT / "data" / "4_model_inputs"
RESULTS_DIR = PROJECT_ROOT / "_03_Models" / "results"
SAVED_MODELS_DIR = PROJECT_ROOT / "_03_Models" / "saved_models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

WindowsPath('d:/NU/semester_8/data_mining/project/phase I')

## 2. Load the shared model inputs

Required files in `data/4_model_inputs/`:

- `train_clean.csv`
- `test_clean.csv`
- `split_metadata.json`

These CSV files contain the saved train/test split before categorical
encoding and numeric scaling.

`TargetEncoded` is the target column:

- 0: Dropout
- 1: Enrolled
- 2: Graduate

Feature types are read from `split_metadata.json` so they remain
consistent with the preprocessing notebook.

No new train/test split is created here.

In [2]:
required_files = [
    "train_clean.csv",
    "test_clean.csv",
    "split_metadata.json",
]

missing_files = [
    name
    for name in required_files
    if not (MODEL_INPUT_DIR / name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing shared model inputs. "
        "Run 02_powerbi_ml_ready_preprocessing.ipynb first.\n"
        f"Missing files: {missing_files}\n"
        f"Expected folder: {MODEL_INPUT_DIR}"
    )

train_df = pd.read_csv(MODEL_INPUT_DIR / "train_clean.csv")
test_df = pd.read_csv(MODEL_INPUT_DIR / "test_clean.csv")

with open(
    MODEL_INPUT_DIR / "split_metadata.json",
    encoding="utf-8",
) as file:
    split_metadata = json.load(file)

TARGET_COLUMN = "TargetEncoded"

for split_name, frame in [
    ("Training", train_df),
    ("Test", test_df),
]:
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(
            f"{split_name} data is missing {TARGET_COLUMN}."
        )

X_train = train_df.drop(columns=TARGET_COLUMN)
X_test = test_df.drop(columns=TARGET_COLUMN)

y_train = train_df[TARGET_COLUMN].astype(int)
y_test = test_df[TARGET_COLUMN].astype(int)

categorical_features = split_metadata["categorical_features"]
numeric_features = split_metadata["numeric_features"]

expected_features = categorical_features + numeric_features

if len(expected_features) != len(set(expected_features)):
    raise ValueError("Feature lists contain duplicate column names.")

for split_name, frame in [
    ("Training", X_train),
    ("Test", X_test),
]:
    if set(frame.columns) != set(expected_features):
        raise ValueError(
            f"{split_name} columns do not match split_metadata.json."
        )

# Keep test columns in the same order as training columns.
X_test = X_test.loc[:, X_train.columns]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))

X_train shape: (3539, 36)
X_test shape: (885, 36)
y_train shape: (3539,)
y_test shape: (885,)
Categorical features: 17
Numeric features: 19


## 3. Train a Decision Tree model

A Decision Tree is an interpretable tree-based classifier. It learns rules that split the data into groups with different outcome patterns.

Because the target classes are imbalanced, `class_weight="balanced"` is used. A small grid search is applied on the training data only to avoid selecting hyperparameters based on the test set.

In [3]:
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            one_hot_encoder,
            categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
    ],
    remainder="drop",
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42,
                class_weight="balanced",
            ),
        ),
    ]
)

param_grid = {
    "model__criterion": ["gini"],
    "model__max_depth": [3, 5, 8, 10, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=1,
    refit=True,
    error_score="raise",
)

grid_search.fit(X_train, y_train)

decision_tree_model = grid_search.best_estimator_

# Preserve the original parameter names in printed and saved results.
best_tree_params = {
    name.replace("model__", "", 1): value
    for name, value in grid_search.best_params_.items()
}

print("Best parameters:", best_tree_params)
print(
    "Best CV Macro-F1:",
    round(grid_search.best_score_, 4),
)


Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 20}
Best CV Macro-F1: 0.6613


## 4. Predict on the test set

In [4]:
y_pred = decision_tree_model.predict(X_test)
y_proba = decision_tree_model.predict_proba(X_test)

print("Predictions generated successfully.")

Predictions generated successfully.


## 5. Evaluate the model

Accuracy alone is not enough because the classes are imbalanced.
We report **Macro-F1**, **Balanced Accuracy**, and **Dropout Recall** alongside the remaining metrics.

This experiment uses first- and second-semester academic features.
Its results therefore do not represent prediction at enrollment time.

In [5]:
labels = [0, 1, 2]
target_names = ["Dropout", "Enrolled", "Graduate"]

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

roc_auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
roc_auc_weighted = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

report_dict = classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)

results = {
    "model": "Decision Tree",
    "accuracy": accuracy,
    "balanced_accuracy": balanced_acc,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1,
    "roc_auc_ovr_macro": roc_auc_macro,
    "roc_auc_ovr_weighted": roc_auc_weighted,
    "dropout_precision": report_dict["Dropout"]["precision"],
    "dropout_recall": report_dict["Dropout"]["recall"],
    "dropout_f1": report_dict["Dropout"]["f1-score"],
}

pd.DataFrame([results]).T

,0
model,Decision Tree
accuracy,0.691525
balanced_accuracy,0.681273
macro_f1,0.665674
weighted_f1,0.712868
roc_auc_ovr_macro,0.841779
roc_auc_ovr_weighted,0.864543
dropout_precision,0.864253
dropout_recall,0.672535
dropout_f1,0.756436


## 6. Classification report and confusion matrix

In [6]:
classification_report_df = pd.DataFrame(report_dict).transpose()
classification_report_df

,precision,recall,f1-score,support
Dropout,0.864253,0.672535,0.756436,284.000000
Enrolled,0.363636,0.654088,0.467416,159.000000
Graduate,0.838624,0.717195,0.773171,442.000000
accuracy,0.691525,0.691525,0.691525,0.691525
macro avg,0.688838,0.681273,0.665674,885.000000
weighted avg,0.761512,0.691525,0.712868,885.000000


In [7]:
conf_matrix = confusion_matrix(y_test, y_pred, labels=labels)
confusion_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[f"actual_{name}" for name in target_names],
    columns=[f"predicted_{name}" for name in target_names],
)
confusion_matrix_df

,predicted_Dropout,predicted_Enrolled,predicted_Graduate
actual_Dropout,191,65,28
actual_Enrolled,22,104,33
actual_Graduate,8,117,317


## 7. Save results and trained model

The saved CSV files will be used later by the final model comparison notebook.

In [8]:
pd.DataFrame([results]).to_csv(RESULTS_DIR / "decision_tree_results.csv", index=False)
classification_report_df.to_csv(RESULTS_DIR / "decision_tree_classification_report.csv")
confusion_matrix_df.to_csv(RESULTS_DIR / "decision_tree_confusion_matrix.csv")

with open(RESULTS_DIR / "decision_tree_best_params.json", "w") as f:
    json.dump(
        {
            "best_params": best_tree_params,
            "best_cv_macro_f1": grid_search.best_score_,
        },
        f,
        indent=2,
    )

joblib.dump(decision_tree_model, SAVED_MODELS_DIR / "decision_tree_model.joblib")

print("Saved:")
print(RESULTS_DIR / "decision_tree_results.csv")
print(RESULTS_DIR / "decision_tree_classification_report.csv")
print(RESULTS_DIR / "decision_tree_confusion_matrix.csv")
print(RESULTS_DIR / "decision_tree_best_params.json")
print(SAVED_MODELS_DIR / "decision_tree_model.joblib")

Saved:
d:\NU\semester_8\data_mining\project\phase I\_03_Models\results\decision_tree_results.csv
d:\NU\semester_8\data_mining\project\phase I\_03_Models\results\decision_tree_classification_report.csv
d:\NU\semester_8\data_mining\project\phase I\_03_Models\results\decision_tree_confusion_matrix.csv
d:\NU\semester_8\data_mining\project\phase I\_03_Models\results\decision_tree_best_params.json
d:\NU\semester_8\data_mining\project\phase I\_03_Models\saved_models\decision_tree_model.joblib


## 8. Short interpretation

The Decision Tree is useful because it is more interpretable than many complex models. However, a single tree may overfit or underperform compared with ensemble tree models such as Random Forest, Gradient Boosting, XGBoost, LightGBM, and CatBoost.

In the final comparison, we will compare this model with Logistic Regression, Naive Bayes, and the remaining models using the same shared test set.